# ContinuityBench v1 offline analysis

This notebook analyzes the frozen Kaggle package and supplied local-run results. It does not call Ollama, use API credentials, or perform inference.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt

# Set this to the Kaggle dataset mount when running on Kaggle.
INPUT_DIR = Path(os.environ.get('CONTINUITYBENCH_INPUT_DIR', '/kaggle/input/continuitybench-v1'))
if not INPUT_DIR.exists():
    INPUT_DIR = Path('.')
print('Input directory:', INPUT_DIR.resolve())

## Load published files

In [ ]:
dataset_path = INPUT_DIR / 'data' / 'continuitybench_v1.jsonl'
per_case_path = INPUT_DIR / 'results' / 'per_case_results.csv'
aggregate_path = INPUT_DIR / 'analysis' / 'aggregate_metrics.csv'
category_path = INPUT_DIR / 'analysis' / 'category_breakdown.csv'
difficulty_path = INPUT_DIR / 'analysis' / 'difficulty_breakdown.csv'

cases = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
per_case = pd.read_csv(per_case_path)
aggregate = pd.read_csv(aggregate_path)
category = pd.read_csv(category_path)
difficulty = pd.read_csv(difficulty_path)
print('Cases:', len(cases), 'Per-case results:', len(per_case))

## Dimensions and schema

In [ ]:
print('Case keys:')
print(sorted(cases[0].keys()))
print('Conversation lengths:', pd.Series([len(c['source_conversation']) for c in cases]).describe())
print(per_case.head(3).to_string(index=False))

## Validate IDs and completeness

In [ ]:
dataset_ids = {c['case_id'] for c in cases}
result_ids = set(per_case['case_id'])
print('Dataset IDs:', len(dataset_ids))
print('Result IDs:', len(result_ids))
print('Missing:', sorted(dataset_ids - result_ids))
print('Unexpected:', sorted(result_ids - dataset_ids))
print('Duplicate result rows:', int(per_case['case_id'].duplicated().sum()))

## Aggregate metrics

In [ ]:
display(aggregate)

## Condition comparison

In [ ]:
mean_table = aggregate.pivot(index='metric', columns='condition', values='mean')
display(mean_table)
mean_table.plot(kind='bar', figsize=(11,5), title='Mean metric values by condition')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

## Category and difficulty breakdowns

In [ ]:
display(category.head(30))
display(difficulty)
for frame, title in [(category, 'Category fact preservation'), (difficulty, 'Difficulty fact preservation')]:
    subset = frame[frame['metric'].eq('fact_preservation')].pivot(index=frame.columns[0], columns='condition', values='mean')
    subset.plot(kind='bar', figsize=(12,5), title=title)
    plt.xticks(rotation=60, ha='right'); plt.tight_layout(); plt.show()

## Interpretation and limitations

`fact_preservation` and `answer_required_fact_coverage` are higher-is-better measures. `stale_fact_error_rate` and `contradiction_rate` are lower-is-better error measures. The implementation uses exact strings and set comparisons, so paraphrases may score as misses. BM25 source retrieval is lexical, not semantic. Zero stale-fact or contradiction error does not establish complete correctness. This is one local run using two models in one Ollama environment, so it does not establish generalization or independent replication. Gold fields in the JSONL are evaluator-only and must not be sent to evaluated models.

## Reproducibility note

All tables above are loaded from the published package; no inference or network access is performed by this notebook.